# 048 — Proyecto: producto ML reproducible

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Proyecto reproducible:** `datos + config + semilla → modelo + métricas`, con un solo
punto de entrada. Datos crudos inmutables (con hash); todo lo derivado se regenera;
configs y semillas versionadas; resultado publicado = commit que lo produjo.

**Cinco fuentes de irreproducibilidad:** datos sin linaje, azar sin semilla, entorno sin
versiones, fuga del protocolo (decidir mirando el test), proceso manual (celdas en orden
mental). Cada una tiene su antídoto estructural.

**Contrato del experimento:** métrica + baseline trivial + intervalo (varias semillas o
bootstrap) + subgrupos + limitaciones. Mejora sin intervalo = fluctuación con buena
prensa. Tests mínimos: features, esquema, determinismo, anti-fuga, humo end-to-end.
La *model card* documenta uso previsto y límites.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (1) Datos: `datos_limpios_DEFINITIVO.csv` sin script de derivación →
guardar crudos inmutables + script + hash. (2) Azar: ninguna semilla a la vista → semillas
en config y reporte multi-semilla. (3) Entorno: sin lockfile → dependencias con versiones
exactas. (4) Protocolo: con 90 celdas y un `.pkl` "v3", nada impide que el test se haya
mirado muchas veces → pipeline con evaluación final separada. (5) Proceso: "ejecutar las
celdas relevantes" es orden mental → un punto de entrada
(`python -m src.pipeline --config configs/base.yaml`).

**Ejercicio 2.** Sí, idénticos: el laboratorio es una función pura de (kind, seed) con
generador de números aleatorios propio — sin estado global, sin I/O, sin dependencia del
reloj. En pipelines reales lo rompen: paralelismo no determinista, orden de lectura de
archivos, operaciones en GPU con reducciones no asociativas, y dependencias sin versionar.

**Ejercicio 3.** (a) A: [14 900, 16 700]; B: [13 100, 15 300] → se solapan en
[14 900, 15 300]. (b) "B parece mejor que A (−1 600, ~10 %), pero la diferencia es del
orden de la variabilidad entre semillas: evidencia sugerente, no concluyente." (c) Más
semillas/particiones (reduce el error estándar de la media) y comparación pareada — mismas
semillas y mismos splits para ambos modelos, evaluando la *diferencia* por semilla, que
elimina la varianza compartida.

**Ejercicio 4.** Ejemplo de model card mínima: *Propósito:* demo educativa de pipeline
integrado (retrieval + agente + política); no apta para decisiones reales. *Datos:*
sintéticos generados por semilla; sin datos personales. *Protocolo:* ejecución
determinista `run_lab("capstone", seed)`. *Métrica:* evidencia estructurada del contrato
JSON; baseline no aplicable a la parte agentica. *Subgrupos:* no evaluados (limitación).
*Límites:* sin persistencia, autenticación ni SLO; `release_gate` exige revisión humana —
los declarados en `limitations` del propio JSON.


In [ ]:
result = run_lab("capstone", seed=48)
assert result["kind"] == "capstone"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 2 — determinismo del laboratorio
r1 = run_lab("capstone", seed=48)
r2 = run_lab("capstone", seed=48)
print("idénticos:", r1 == r2)
print("gate:", r1["result"]["release_gate"])
# Función pura de (kind, seed): mismo input → mismo dict completo.
# En producción lo rompen paralelismo, GPU no determinista y entornos sin lockear.


In [ ]:
# Ejercicio 3 — intervalos y conclusión honesta
media_a, desv_a = 15_800, 900
media_b, desv_b = 14_200, 1_100
int_a = (media_a - desv_a, media_a + desv_a)
int_b = (media_b - desv_b, media_b + desv_b)
solapan = int_a[0] <= int_b[1] and int_b[0] <= int_a[1]
print(f"A: {int_a}  B: {int_b}  solapan: {solapan}")
print("conclusión: mejora probable de B, no concluyente con 5 semillas;")
print("siguiente paso: comparación pareada por semilla + más particiones")


## Reflexión

1. El laboratorio `capstone` integra recuperación, agente y política de permisos, y
   termina en `release_gate: human_review_required`. ¿Qué equivalente tiene ese gate en un
   proyecto ML clásico y qué evidencia debería revisar el humano antes de aprobar?
2. De las cinco fuentes de irreproducibilidad, ¿cuáles elimina el diseño
   `run_lab(kind, seed)` de este repositorio y cuáles quedan fuera de su alcance?
3. Tu gradient boosting mejora a la logística en 1 600 unidades de costo, con intervalos
   que se solapan. Defiende ambas decisiones posibles (desplegar / no desplegar) y señala
   qué dato adicional zanjaría la discusión.
